# Heritage Short: Wan 2.2 TI2V-5B image-to-video (Kaggle, 2x T4)

Reads `jobs.json` from the attached dataset (made by `python -m pipeline.shots export`),
generates one clip per job, and writes `<shot>__<key>.mp4` plus `runs_kaggle.jsonl` to
`/kaggle/working`. Download the output (or `kaggle kernels output ...`) and run
`python -m pipeline.shots import <folder>` locally.

**Recommended workflow (prove one shot first):**
1. Start with `WAN_ONLY=s02` (single job), `640x640` (divisible by 32), `49` frames, `30` steps.
2. The worker uses `WanImageToVideoPipeline` in `bfloat16`, `enable_model_cpu_offload()`, and `vae.enable_tiling()`.
3. On OOM it automatically retries `512x512` (49 frames) then `512x512` (33 frames). If one 49-frame clip exceeds 20 GPU minutes, abort and use the Ken Burns fallback for remaining shots (document in `FAILURES.md`).
4. If `WanImageToVideoPipeline` fails to load `TI2V-5B`, fall back to Wan2.2's `generate.py --task ti2v-5B --image ... --offload_model True --convert_model_dtype --t5_cpu`.

In [ ]:
!pip -q install -U git+https://github.com/huggingface/diffusers transformers accelerate ftfy imageio imageio-ffmpeg sentencepiece

In [ ]:
import glob, json, os, subprocess, sys, pathlib
cands = glob.glob('/kaggle/input/**/jobs.json', recursive=True)
assert cands, 'Attach the heritage-short-jobs dataset (Add data -> Your datasets)'
JOBS = pathlib.Path(cands[0]).parent
jobs = json.load(open(JOBS / 'jobs.json'))['jobs']
print(len(jobs), 'jobs from', JOBS)
for j in jobs: print(j['shot_id'], j['width'], 'x', j['height'], j['frames'], 'frames |', j['prompt'][:60])

In [ ]:
WORKER = r"""
import json, os, sys, time, torch, datetime
from diffusers import AutoencoderKLWan, WanImageToVideoPipeline
from diffusers.utils import export_to_video, load_image

jobs_dir, out_dir, shard, nshards = sys.argv[1], sys.argv[2], int(sys.argv[3]), int(sys.argv[4])
jobs = json.load(open(os.path.join(jobs_dir, 'jobs.json')))['jobs']
only = [s.strip() for s in os.environ.get('WAN_ONLY', '').split(',') if s.strip()]
if only:
    jobs = [j for j in jobs if j['shot_id'] in only]
jobs = jobs[shard::nshards]
if not jobs: sys.exit(0)

MODEL = os.environ.get('WAN_MODEL', 'Wan-AI/Wan2.2-TI2V-5B-Diffusers')
MAX_FRAMES = int(os.environ.get('WAN_MAX_FRAMES', '49'))
MAX_GPU_S = float(os.environ.get('WAN_MAX_GPU_S', '1200'))
dtype = torch.float16 if os.environ.get('WAN_FP16') == '1' else torch.bfloat16

t0 = time.time()
vae = AutoencoderKLWan.from_pretrained(MODEL, subfolder='vae', torch_dtype=torch.float32)
pipe = WanImageToVideoPipeline.from_pretrained(MODEL, vae=vae, torch_dtype=dtype)
# FAILURES.md F6: enable_model_cpu_offload() keeps the 10 GB transformer resident on a 15 GB T4,
# which OOMs at step 0 (and every retry, since the hooks leave it on the GPU). Sequential
# offload streams layer by layer: slower, but fits. WAN_OFFLOAD=model restores the old mode.
if os.environ.get('WAN_OFFLOAD', 'sequential') == 'model':
    pipe.enable_model_cpu_offload()
else:
    pipe.enable_sequential_cpu_offload()
try: pipe.vae.enable_tiling()
except Exception: pass
load_s = time.time() - t0

def round32(v): return max(32, (v // 32) * 32)

log = open(os.path.join(out_dir, 'runs_kaggle.jsonl'), 'a')
for j in jobs:
    out = os.path.join(out_dir, j['output'])
    w0, h0 = round32(min(j['width'], 640)), round32(min(j['height'], 640))
    f0 = min(j['frames'], MAX_FRAMES)
    attempts = [(w0, h0, f0), (512, 512, f0), (512, 512, min(f0, 33))]
    rec = {
        'ts': datetime.datetime.utcnow().isoformat(timespec='seconds') + 'Z',
        'run_id': 'kaggle-' + os.environ.get('KAGGLE_KERNEL_RUN_TYPE', 'run'),
        'stage': 'shots', 'item': j['shot_id'], 'provider': 'kaggle', 'model': MODEL, 'seed': j['seed'],
        'params': {k: j[k] for k in ('width', 'height', 'frames', 'fps', 'steps', 'guidance', 'prompt')},
        'cache_key': j['key'], 'gpu': torch.cuda.get_device_name(0), 'dtype': str(dtype),
        'model_load_s': round(load_s, 1), 'cost_usd': 0.0
    }
    t = time.time()
    torch.cuda.reset_peak_memory_stats()
    for w, h, nframes in attempts:
        try:
            img = load_image(os.path.join(jobs_dir, j['image'])).resize((w, h))
            frames = pipe(image=img, prompt=j['prompt'], negative_prompt=j['negative_prompt'],
                          height=h, width=w, num_frames=nframes,
                          num_inference_steps=j['steps'], guidance_scale=j['guidance'],
                          generator=torch.Generator('cpu').manual_seed(j['seed'])).frames[0]
            export_to_video(frames, out, fps=j['fps'])
            rec.update(status='ok', output=j['output'], used_size=f'{w}x{h}', used_frames=nframes)
            break
        except Exception as e:
            import gc; gc.collect(); torch.cuda.empty_cache()
            try: pipe.maybe_free_model_hooks()
            except Exception: pass
            is_oom = 'out of memory' in str(e).lower()
            rec.update(status='oom' if is_oom else 'fatal_error', error=str(e)[:500])
            if not is_oom:
                break
    dt = time.time() - t
    rec.update(latency_s=round(dt, 1), gpu_s=round(dt, 1), peak_vram_gb=round(torch.cuda.max_memory_allocated() / 1e9, 2))
    log.write(json.dumps(rec) + '\n'); log.flush()
    print(j['shot_id'], rec['status'], f'{dt:.0f}s', 'vram', rec['peak_vram_gb'], 'GB', flush=True)
    if dt > MAX_GPU_S:
        print('Exceeded 20 GPU minute budget per clip; stopping early.', flush=True)
        break
"""
open('/kaggle/working/wan_worker.py', 'w').write(WORKER)

In [ ]:
import os, torch
# Proof run: set WAN_ONLY='s02' first; clear it to run all approved shots in parallel across GPUs.
env_extra = {'WAN_ONLY': os.environ.get('WAN_ONLY', 's02')}
n = max(1, torch.cuda.device_count())
print('GPUs:', n, '| WAN_ONLY:', env_extra['WAN_ONLY'] or 'ALL')
procs = [subprocess.Popen([sys.executable, '/kaggle/working/wan_worker.py', str(JOBS), '/kaggle/working', str(i), str(n)],
                          env={**os.environ, **env_extra, 'CUDA_VISIBLE_DEVICES': str(i)}) for i in range(n)]
codes = [p.wait() for p in procs]
print('exit codes', codes)

In [ ]:
!rm -f /kaggle/working/wan_worker.py
!ls -la /kaggle/working && tail -n 20 /kaggle/working/runs_kaggle.jsonl